In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.ensemble import RandomForestClassifier

numeric_features = [
    "age",
    "income"
]

categorical_features = [
    "city",
    "education"
]

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_features
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_features
    )
])

pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=42
        )
    )
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

In [ ]:
param_grid = {
    "model__n_estimators": [
        100,
        200
    ],
    "model__max_depth": [
        None,
        10,
        20
    ],
    "model__min_samples_split": [
        2,
        5
    ]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-2
)

grid.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(grid.best_params_)

print("Best CV score:")
print(grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

test_score = best_model.score(
    X_test,
    y_test
)

print("Test score:", test_score)

                  RAW DATA
                     │
                     ↓
              Train / Test Split
                /           \
               ↓             ↓
           TRAINING         TEST
               │
               ↓
      ┌──────────────────┐
      │ Feature Types    │
      └────────┬─────────┘
               │
       ┌───────┴────────┐
       ↓                ↓
 Numerical          Categorical
       ↓                ↓
 Imputation          Imputation
       ↓                ↓
 Scaling            Encoding
       └───────┬────────┘
               ↓
          Combined Data
               ↓
             Model
               ↓
       Cross Validation
               ↓
     Hyperparameter Tuning
               ↓
        Best Pipeline
               ↓
        Unseen Test Data
               ↓
       Final Evaluation

In [1]:
import joblib